In [52]:
import os, gc, json, warnings
import pandas as pd
import numpy as np
from datetime import datetime
from datasets import load_dataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler
from imblearn.under_sampling import RandomUnderSampler, TomekLinks, NearMiss, OneSidedSelection, NeighbourhoodCleaningRule
from imblearn.over_sampling import SMOTE, ADASYN, BorderlineSMOTE, RandomOverSampler
from imblearn.combine import SMOTETomek, SMOTEENN
from sklearn.feature_selection import SelectKBest, mutual_info_classif, chi2, f_classif, VarianceThreshold, RFE, SelectFromModel
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.decomposition import PCA, TruncatedSVD, KernelPCA
from sklearn.linear_model import LogisticRegression, LassoCV
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA

try:
    import umap
    UMAP_AVAILABLE = True
except:
    UMAP_AVAILABLE = False

warnings.filterwarnings('ignore')
print("✅ Imports OK")

✅ Imports OK


In [53]:
CONFIG = {
    'sample_size': 1_000_000,
    'test_size': 0.15,
    'val_size': 0.15,
    'n_features_to_select': 10,
    'correlation_threshold': 0.95,
    'n_components_pca': 8,
    'n_components_lda': 1,
    'n_components_svd': 8,
    'n_components_umap': 8,
    
    # Kernel PCA - TRÈS LOURD, désactivé par défaut
    'enable_kernel_pca': False,  # Mettre True si vous avez >16GB RAM
    'kernel_pca_sample_size': 5000,  # Réduit de 50K à 5K pour ordinateurs faibles
    
    'output_dir': './data_variants/',
    'random_state': 42,
    'skip_existing': True,
}
os.makedirs(CONFIG['output_dir'], exist_ok=True)
print(f"📁 Output: {CONFIG['output_dir']}")
if CONFIG['enable_kernel_pca']:
    print("⚠️ Kernel PCA activé - cela peut prendre du temps sur PC peu puissant")
else:
    print("ℹ️ Kernel PCA désactivé (trop lourd). Activez dans CONFIG si besoin.")

📁 Output: ./data_variants/
ℹ️ Kernel PCA désactivé (trop lourd). Activez dans CONFIG si besoin.


In [54]:
def save_dataset(X_train, y_train, X_val, y_val, X_test, y_test, name, meta):
    
    if CONFIG['skip_existing'] and variant_exists(name):
        print(f"⏭️  {name}: Déjà existant, skip")
        return True
    
    path = os.path.join(CONFIG['output_dir'], name)
    os.makedirs(path, exist_ok=True)
    df_tr = X_train.copy()
    df_tr['isFraud'] = y_train.values if hasattr(y_train, 'values') else y_train
    df_v = X_val.copy()
    df_v['isFraud'] = y_val.values if hasattr(y_val, 'values') else y_val
    df_te = X_test.copy()
    df_te['isFraud'] = y_test.values if hasattr(y_test, 'values') else y_test
    df_tr.to_parquet(f'{path}/train.parquet', index=False)
    df_v.to_parquet(f'{path}/val.parquet', index=False)
    df_te.to_parquet(f'{path}/test.parquet', index=False)
    y_tr = y_train.values if hasattr(y_train, 'values') else y_train
    y_v = y_val.values if hasattr(y_val, 'values') else y_val
    y_te = y_test.values if hasattr(y_test, 'values') else y_test
    meta['shapes'] = {'train': list(df_tr.shape), 'val': list(df_v.shape), 'test': list(df_te.shape)}
    meta['fraud_distribution'] = {
        'train': {str(k): int(v) for k,v in zip(*np.unique(y_tr, return_counts=True))},
        'val': {str(k): int(v) for k,v in zip(*np.unique(y_v, return_counts=True))},
        'test': {str(k): int(v) for k,v in zip(*np.unique(y_te, return_counts=True))}
    }
    meta['created_at'] = datetime.now().isoformat()
    with open(f'{path}/metadata.json', 'w') as f: json.dump(meta, f, indent=2)
    fraud = int(np.sum(y_tr))
    print(f"✅ {name}: {df_tr.shape}, Fraud {fraud} ({fraud/df_tr.shape[0]*100:.1f}%)")
    
    return False

def scale_features(X_tr, X_v, X_te):
    sc = RobustScaler()
    return (
        pd.DataFrame(sc.fit_transform(X_tr), columns=X_tr.columns, index=X_tr.index),
        pd.DataFrame(sc.transform(X_v), columns=X_v.columns, index=X_v.index),
        pd.DataFrame(sc.transform(X_te), columns=X_te.columns, index=X_te.index)
    )

def remove_correlated_features(X, threshold=0.95):
    corr = X.corr().abs()
    upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
    to_drop = [col for col in upper.columns if any(upper[col] > threshold)]
    return X.drop(columns=to_drop), to_drop

def variant_exists(name):
    '''Vérifie si une variante existe déjà avec tous ses fichiers'''
    path = os.path.join(CONFIG['output_dir'], name)
    if not os.path.exists(path):
        return False
    required_files = ['train.parquet', 'val.parquet', 'test.parquet', 'metadata.json']
    return all(os.path.exists(os.path.join(path, f)) for f in required_files)

print("✅ Fonctions OK")

✅ Fonctions OK


In [55]:
print("🔄 Chargement...")
df = load_dataset("CiferAI/Cifer-Fraud-Detection-Dataset-AF", split="train").to_pandas()
print(f"📊 Original: {df.shape}")
if CONFIG['sample_size'] and CONFIG['sample_size'] < len(df):
    df = df.groupby('isFraud', group_keys=False).apply(
        lambda x: x.sample(n=min(len(x), CONFIG['sample_size']//2), random_state=42)
    ).reset_index(drop=True)
    print(f"📊 Échantillon: {df.shape}")

🔄 Chargement...
📊 Original: (21000000, 11)
📊 Échantillon: (527470, 11)


In [56]:
print("🔧 Feature engineering...")
df["hour"] = df["step"] % 24
df["day"] = df["step"] // 24
df = pd.get_dummies(df, columns=["type"], prefix="type")
df["delta_sender"] = df["newbalanceOrig"] - df["oldbalanceOrg"]
df["abs_delta_sender"] = df["delta_sender"].abs()
df["balance_sender_error"] = df["delta_sender"] + df["amount"]
df["abs_balance_sender_error"] = df["balance_sender_error"].abs()
df["delta_receiver"] = df["newbalanceDest"] - df["oldbalanceDest"]
df["abs_delta_receiver"] = df["delta_receiver"].abs()
df["balance_receiver_error"] = df["delta_receiver"] - df["amount"]
df["abs_balance_receiver_error"] = df["balance_receiver_error"].abs()
df["amount_over_old"] = df["amount"] / (df["oldbalanceOrg"] + 1)
df["amount_over_new"] = df["amount"] / (df["newbalanceOrig"] + 1)
df["amount_over_total"] = df["amount"] / (df["oldbalanceOrg"] + df["newbalanceOrig"] + 1)
df = df.drop(columns=["nameOrig", "nameDest", "isFlaggedFraud"])
for col in df.columns:
    if col != 'isFraud':
        df[col] = df[col].astype('float64')
print(f"✅ {df.shape[1]} features")

🔧 Feature engineering...
✅ 25 features


In [57]:
X = df.drop(columns=["isFraud"])
y = df["isFraud"]
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=(CONFIG['test_size']+CONFIG['val_size']), stratify=y, random_state=42)
val_ratio = CONFIG['val_size'] / (CONFIG['test_size'] + CONFIG['val_size'])
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=(1-val_ratio), stratify=y_temp, random_state=42)
print(f"Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")
del df, X, y, X_temp, y_temp
gc.collect()
X_train_scaled, X_val_scaled, X_test_scaled = scale_features(X_train, X_val, X_test)

Train: (369229, 24), Val: (79120, 24), Test: (79121, 24)


In [58]:
print("\n🚀 GÉNÉRATION DE TOUTES LES VARIANTES\n")
cnt = 0
skipped = 0
generated = 0

# BASELINE
cnt += 1
print(f"[{cnt}] Baseline")
if save_dataset(X_train_scaled, y_train, X_val_scaled, y_val, X_test_scaled, y_test, 'baseline',
    {'sampling': 'none', 'feature_selection': 'all', 'dim_reduction': 'none'}):
    skipped += 1
else:
    generated += 1
gc.collect()

# SAMPLING (11 variantes)
samplers = {
    'undersampling_random': RandomUnderSampler(random_state=42),
    'undersampling_tomek': TomekLinks(),
    'undersampling_nearmiss': NearMiss(version=1),
    'undersampling_onesided': OneSidedSelection(random_state=42),
    'undersampling_ncr': NeighbourhoodCleaningRule(),
    'oversampling_random': RandomOverSampler(random_state=42),
    'oversampling_smote': SMOTE(random_state=42, k_neighbors=3),
    'oversampling_adasyn': ADASYN(random_state=42, n_neighbors=3),
    'oversampling_borderline': BorderlineSMOTE(random_state=42, k_neighbors=3),
    'combined_smote_tomek': SMOTETomek(random_state=42),
    'combined_smote_enn': SMOTEENN(random_state=42)
}
for name, sampler in samplers.items():
    try:
        cnt += 1
        print(f"[{cnt}] {name}")
        
        if CONFIG['skip_existing'] and variant_exists(name):
            print(f"⏭️ {name}: Déjà existant, skip")
            skipped += 1
            continue
        
        X_r, y_r = sampler.fit_resample(X_train, y_train)
        if not isinstance(X_r, pd.DataFrame): X_r = pd.DataFrame(X_r, columns=X_train.columns)
        if not isinstance(y_r, pd.Series): y_r = pd.Series(y_r, name='isFraud')
        X_s, V_s, T_s = scale_features(X_r, X_val, X_test)
        save_dataset(X_s, y_r, V_s, y_val, T_s, y_test, name,
            {'sampling': name, 'feature_selection': 'all', 'dim_reduction': 'none'})
        generated += 1
        del X_r, y_r, X_s, V_s, T_s
        gc.collect()
    except Exception as e:
        print(f"❌ {name}: {e}")

# FEATURE SELECTION (12 variantes)
cnt += 1
print(f"[{cnt}] ExtraTrees")
if not (CONFIG['skip_existing'] and variant_exists('feature_selection_extratrees')):
    et = ExtraTreesClassifier(n_estimators=50, max_depth=10, random_state=42, n_jobs=-1)
    et.fit(X_train_scaled, y_train)
    top_et = pd.DataFrame({'f': X_train_scaled.columns, 'i': et.feature_importances_}).nlargest(10, 'i')['f'].tolist()
    save_dataset(X_train_scaled[top_et], y_train, X_val_scaled[top_et], y_val, X_test_scaled[top_et], y_test,
        'feature_selection_extratrees', {'sampling': 'none', 'feature_selection': 'ExtraTrees', 'dim_reduction': 'none'})
    generated += 1
    del et
else:
    print(f"⏭️ feature_selection_extratrees: Déjà existant, skip")
    skipped += 1
gc.collect()

cnt += 1
print(f"[{cnt}] RandomForest")
if not (CONFIG['skip_existing'] and variant_exists('feature_selection_randomforest')):
    rf = RandomForestClassifier(n_estimators=50, max_depth=10, random_state=42, n_jobs=-1)
    rf.fit(X_train_scaled, y_train)
    top_rf = pd.DataFrame({'f': X_train_scaled.columns, 'i': rf.feature_importances_}).nlargest(10, 'i')['f'].tolist()
    save_dataset(X_train_scaled[top_rf], y_train, X_val_scaled[top_rf], y_val, X_test_scaled[top_rf], y_test,
        'feature_selection_randomforest', {'sampling': 'none', 'feature_selection': 'RandomForest', 'dim_reduction': 'none'})
    generated += 1
    del rf
else:
    print(f"⏭️ feature_selection_randomforest: Déjà existant, skip")
    skipped += 1
gc.collect()

cnt += 1
print(f"[{cnt}] Mutual Info")
if not (CONFIG['skip_existing'] and variant_exists('feature_selection_mutual_info')):
    sel_mi = SelectKBest(mutual_info_classif, k=10)
    sel_mi.fit(X_train_scaled, y_train)
    feat_mi = X_train_scaled.columns[sel_mi.get_support()].tolist()
    save_dataset(pd.DataFrame(sel_mi.transform(X_train_scaled), columns=feat_mi, index=X_train_scaled.index), y_train,
        pd.DataFrame(sel_mi.transform(X_val_scaled), columns=feat_mi, index=X_val_scaled.index), y_val,
        pd.DataFrame(sel_mi.transform(X_test_scaled), columns=feat_mi, index=X_test_scaled.index), y_test,
        'feature_selection_mutual_info', {'sampling': 'none', 'feature_selection': 'MutualInfo', 'dim_reduction': 'none'})
    generated += 1
    del sel_mi
else:
    print(f"⏭️ feature_selection_mutual_info: Déjà existant, skip")
    skipped += 1
gc.collect()

cnt += 1
print(f"[{cnt}] Chi2")
if not (CONFIG['skip_existing'] and variant_exists('feature_selection_chi2')):
    X_pos = X_train_scaled - X_train_scaled.min() + 1e-10
    sel_chi = SelectKBest(chi2, k=10)
    sel_chi.fit(X_pos, y_train)
    feat_chi = X_train_scaled.columns[sel_chi.get_support()].tolist()
    save_dataset(pd.DataFrame(sel_chi.transform(X_pos), columns=feat_chi, index=X_train_scaled.index), y_train,
        pd.DataFrame(sel_chi.transform(X_val_scaled-X_val_scaled.min()+1e-10), columns=feat_chi, index=X_val_scaled.index), y_val,
        pd.DataFrame(sel_chi.transform(X_test_scaled-X_test_scaled.min()+1e-10), columns=feat_chi, index=X_test_scaled.index), y_test,
        'feature_selection_chi2', {'sampling': 'none', 'feature_selection': 'Chi2', 'dim_reduction': 'none'})
    generated += 1
    del sel_chi, X_pos
else:
    print(f"⏭️ feature_selection_chi2: Déjà existant, skip")
    skipped += 1
gc.collect()

cnt += 1
print(f"[{cnt}] ANOVA")
if not (CONFIG['skip_existing'] and variant_exists('feature_selection_anova')):
    sel_an = SelectKBest(f_classif, k=10)
    sel_an.fit(X_train_scaled, y_train)
    feat_an = X_train_scaled.columns[sel_an.get_support()].tolist()
    save_dataset(pd.DataFrame(sel_an.transform(X_train_scaled), columns=feat_an, index=X_train_scaled.index), y_train,
        pd.DataFrame(sel_an.transform(X_val_scaled), columns=feat_an, index=X_val_scaled.index), y_val,
        pd.DataFrame(sel_an.transform(X_test_scaled), columns=feat_an, index=X_test_scaled.index), y_test,
        'feature_selection_anova', {'sampling': 'none', 'feature_selection': 'ANOVA', 'dim_reduction': 'none'})
    generated += 1
    del sel_an
else:
    print(f"⏭️ feature_selection_anova: Déjà existant, skip")
    skipped += 1
gc.collect()

cnt += 1
print(f"[{cnt}] Variance")
if not (CONFIG['skip_existing'] and variant_exists('feature_selection_variance')):
    sel_var = VarianceThreshold(0.1)
    sel_var.fit(X_train_scaled)
    feat_var = X_train_scaled.columns[sel_var.get_support()].tolist()
    if len(feat_var) > 10: feat_var = X_train_scaled[feat_var].var().nlargest(10).index.tolist()
    save_dataset(X_train_scaled[feat_var], y_train, X_val_scaled[feat_var], y_val, X_test_scaled[feat_var], y_test,
        'feature_selection_variance', {'sampling': 'none', 'feature_selection': 'Variance', 'dim_reduction': 'none'})
    generated += 1
    del sel_var
else:
    print(f"⏭️ feature_selection_variance: Déjà existant, skip")
    skipped += 1
gc.collect()

cnt += 1
print(f"[{cnt}] RFE")
if not (CONFIG['skip_existing'] and variant_exists('feature_selection_rfe')):
    rfe = RFE(LogisticRegression(max_iter=500, random_state=42), n_features_to_select=10)
    rfe.fit(X_train_scaled, y_train)
    feat_rfe = X_train_scaled.columns[rfe.get_support()].tolist()
    save_dataset(pd.DataFrame(rfe.transform(X_train_scaled), columns=feat_rfe, index=X_train_scaled.index), y_train,
        pd.DataFrame(rfe.transform(X_val_scaled), columns=feat_rfe, index=X_val_scaled.index), y_val,
        pd.DataFrame(rfe.transform(X_test_scaled), columns=feat_rfe, index=X_test_scaled.index), y_test,
        'feature_selection_rfe', {'sampling': 'none', 'feature_selection': 'RFE', 'dim_reduction': 'none'})
    generated += 1
    del rfe
else:
    print(f"⏭️ feature_selection_rfe: Déjà existant, skip")
    skipped += 1
gc.collect()

cnt += 1
print(f"[{cnt}] L1-Lasso")
if not (CONFIG['skip_existing'] and variant_exists('feature_selection_l1_lasso')):
    lasso = LassoCV(cv=3, random_state=42, max_iter=1000, n_jobs=-1)
    lasso.fit(X_train_scaled, y_train)
    feat_l1 = pd.Series(np.abs(lasso.coef_), index=X_train_scaled.columns).nlargest(10).index.tolist()
    save_dataset(X_train_scaled[feat_l1], y_train, X_val_scaled[feat_l1], y_val, X_test_scaled[feat_l1], y_test,
        'feature_selection_l1_lasso', {'sampling': 'none', 'feature_selection': 'L1_Lasso', 'dim_reduction': 'none'})
    generated += 1
    del lasso
else:
    print(f"⏭️ feature_selection_l1_lasso: Déjà existant, skip")
    skipped += 1
gc.collect()

cnt += 1
print(f"[{cnt}] Correlation Filter")
if not (CONFIG['skip_existing'] and variant_exists('feature_selection_correlation')):
    X_uncorr, dropped = remove_correlated_features(X_train_scaled, CONFIG['correlation_threshold'])
    print(f"  Dropped {len(dropped)} correlated features")
    if X_uncorr.shape[1] > 10:
        top_var = X_uncorr.var().nlargest(10).index.tolist()
        X_uncorr = X_uncorr[top_var]
    feat_corr = X_uncorr.columns.tolist()
    save_dataset(X_uncorr, y_train, X_val_scaled[feat_corr], y_val, X_test_scaled[feat_corr], y_test,
        'feature_selection_correlation', {'sampling': 'none', 'feature_selection': 'Correlation', 'dim_reduction': 'none'})
    generated += 1
    del X_uncorr
else:
    print(f"⏭️ feature_selection_correlation: Déjà existant, skip")
    skipped += 1
gc.collect()

cnt += 1
print(f"[{cnt}] Permutation Importance")
if not (CONFIG['skip_existing'] and variant_exists('feature_selection_permutation')):
    rf_perm = RandomForestClassifier(n_estimators=30, max_depth=8, random_state=42, n_jobs=-1)
    rf_perm.fit(X_train_scaled, y_train)
    perm = permutation_importance(rf_perm, X_train_scaled, y_train, n_repeats=5, random_state=42, n_jobs=-1)
    feat_perm = pd.Series(perm.importances_mean, index=X_train_scaled.columns).nlargest(10).index.tolist()
    save_dataset(X_train_scaled[feat_perm], y_train, X_val_scaled[feat_perm], y_val, X_test_scaled[feat_perm], y_test,
        'feature_selection_permutation', {'sampling': 'none', 'feature_selection': 'Permutation', 'dim_reduction': 'none'})
    generated += 1
    del rf_perm, perm
else:
    print(f"⏭️ feature_selection_permutation: Déjà existant, skip")
    skipped += 1
gc.collect()

cnt += 1
print(f"[{cnt}] Combined Statistical")
if not (CONFIG['skip_existing'] and variant_exists('feature_selection_combined_stat')):
    s_mi = SelectKBest(mutual_info_classif, k='all')
    s_mi.fit(X_train_scaled, y_train)
    sc_mi = pd.Series(s_mi.scores_, index=X_train_scaled.columns)
    sc_var = X_train_scaled.var()
    s_an = SelectKBest(f_classif, k='all')
    s_an.fit(X_train_scaled, y_train)
    sc_an = pd.Series(s_an.scores_, index=X_train_scaled.columns)
    sc_mi_n = (sc_mi - sc_mi.min()) / (sc_mi.max() - sc_mi.min())
    sc_var_n = (sc_var - sc_var.min()) / (sc_var.max() - sc_var.min())
    sc_an_n = (sc_an - sc_an.min()) / (sc_an.max() - sc_an.min())
    sc_comb = (sc_mi_n + sc_var_n + sc_an_n) / 3
    feat_comb = sc_comb.nlargest(10).index.tolist()
    save_dataset(X_train_scaled[feat_comb], y_train, X_val_scaled[feat_comb], y_val, X_test_scaled[feat_comb], y_test,
        'feature_selection_combined_stat', {'sampling': 'none', 'feature_selection': 'Combined_Statistical', 'dim_reduction': 'none'})
    generated += 1
    del s_mi, s_an
else:
    print(f"⏭️ feature_selection_combined_stat: Déjà existant, skip")
    skipped += 1
gc.collect()

# DIMENSIONALITY REDUCTION (4-5 variantes selon config)
cnt += 1
print(f"[{cnt}] PCA")
if not (CONFIG['skip_existing'] and variant_exists('dim_reduction_pca')):
    pca = PCA(n_components=8, random_state=42)
    Xtr = pd.DataFrame(pca.fit_transform(X_train_scaled), columns=[f'PC{i+1}' for i in range(8)], index=X_train_scaled.index)
    Xv = pd.DataFrame(pca.transform(X_val_scaled), columns=[f'PC{i+1}' for i in range(8)], index=X_val_scaled.index)
    Xte = pd.DataFrame(pca.transform(X_test_scaled), columns=[f'PC{i+1}' for i in range(8)], index=X_test_scaled.index)
    save_dataset(Xtr, y_train, Xv, y_val, Xte, y_test, 'dim_reduction_pca',
        {'sampling': 'none', 'feature_selection': 'none', 'dim_reduction': 'PCA'})
    generated += 1
    del pca, Xtr, Xv, Xte
else:
    print(f"⏭️ dim_reduction_pca: Déjà existant, skip")
    skipped += 1
gc.collect()

cnt += 1
print(f"[{cnt}] LDA")
if not (CONFIG['skip_existing'] and variant_exists('dim_reduction_lda')):
    lda = LDA(n_components=1)
    Xtr = pd.DataFrame(lda.fit_transform(X_train_scaled, y_train), columns=['LD1'], index=X_train_scaled.index)
    Xv = pd.DataFrame(lda.transform(X_val_scaled), columns=['LD1'], index=X_val_scaled.index)
    Xte = pd.DataFrame(lda.transform(X_test_scaled), columns=['LD1'], index=X_test_scaled.index)
    save_dataset(Xtr, y_train, Xv, y_val, Xte, y_test, 'dim_reduction_lda',
        {'sampling': 'none', 'feature_selection': 'none', 'dim_reduction': 'LDA'})
    generated += 1
    del lda, Xtr, Xv, Xte
else:
    print(f"⏭️ dim_reduction_lda: Déjà existant, skip")
    skipped += 1
gc.collect()

cnt += 1
print(f"[{cnt}] Truncated SVD")
if not (CONFIG['skip_existing'] and variant_exists('dim_reduction_svd')):
    svd = TruncatedSVD(n_components=8, random_state=42)
    Xtr = pd.DataFrame(svd.fit_transform(X_train_scaled), columns=[f'SVD{i+1}' for i in range(8)], index=X_train_scaled.index)
    Xv = pd.DataFrame(svd.transform(X_val_scaled), columns=[f'SVD{i+1}' for i in range(8)], index=X_val_scaled.index)
    Xte = pd.DataFrame(svd.transform(X_test_scaled), columns=[f'SVD{i+1}' for i in range(8)], index=X_test_scaled.index)
    save_dataset(Xtr, y_train, Xv, y_val, Xte, y_test, 'dim_reduction_svd',
        {'sampling': 'none', 'feature_selection': 'none', 'dim_reduction': 'TruncatedSVD'})
    generated += 1
    del svd, Xtr, Xv, Xte
else:
    print(f"⏭️ dim_reduction_svd: Déjà existant, skip")
    skipped += 1
gc.collect()

# KERNEL PCA - OPTIONNEL (désactivé par défaut)
if CONFIG['enable_kernel_pca']:
    cnt += 1
    print(f"[{cnt}] Kernel PCA (RBF) - échantillon {CONFIG['kernel_pca_sample_size']}")
    if not (CONFIG['skip_existing'] and variant_exists('dim_reduction_kernel_pca')):
        print("  ⚠️ Cela peut prendre plusieurs minutes...")
        sample_size = min(CONFIG['kernel_pca_sample_size'], len(X_train_scaled))
        sample_idx = np.random.choice(len(X_train_scaled), sample_size, replace=False)
        X_sample = X_train_scaled.iloc[sample_idx]
        kpca = KernelPCA(n_components=8, kernel='rbf', random_state=42, n_jobs=-1)
        kpca.fit(X_sample)
        Xtr = pd.DataFrame(kpca.transform(X_train_scaled), columns=[f'KPCA{i+1}' for i in range(8)], index=X_train_scaled.index)
        Xv = pd.DataFrame(kpca.transform(X_val_scaled), columns=[f'KPCA{i+1}' for i in range(8)], index=X_val_scaled.index)
        Xte = pd.DataFrame(kpca.transform(X_test_scaled), columns=[f'KPCA{i+1}' for i in range(8)], index=X_test_scaled.index)
        save_dataset(Xtr, y_train, Xv, y_val, Xte, y_test, 'dim_reduction_kernel_pca',
            {'sampling': 'none', 'feature_selection': 'none', 'dim_reduction': 'KernelPCA_RBF'})
        generated += 1
        del kpca, Xtr, Xv, Xte, X_sample
    else:
        print(f"⏭️ dim_reduction_kernel_pca: Déjà existant, skip")
        skipped += 1
    gc.collect()
else:
    print(f"⏭️ Kernel PCA désactivé (activez dans CONFIG si besoin)")

if UMAP_AVAILABLE:
    cnt += 1
    print(f"[{cnt}] UMAP")
    if not (CONFIG['skip_existing'] and variant_exists('dim_reduction_umap')):
        um = umap.UMAP(n_components=8, n_neighbors=15, random_state=42)
        Xtr = pd.DataFrame(um.fit_transform(X_train_scaled, y=y_train), columns=[f'U{i+1}' for i in range(8)], index=X_train_scaled.index)
        Xv = pd.DataFrame(um.transform(X_val_scaled), columns=[f'U{i+1}' for i in range(8)], index=X_val_scaled.index)
        Xte = pd.DataFrame(um.transform(X_test_scaled), columns=[f'U{i+1}' for i in range(8)], index=X_test_scaled.index)
        save_dataset(Xtr, y_train, Xv, y_val, Xte, y_test, 'dim_reduction_umap',
            {'sampling': 'none', 'feature_selection': 'none', 'dim_reduction': 'UMAP'})
        generated += 1
        del um, Xtr, Xv, Xte
    else:
        print(f"⏭️ dim_reduction_umap: Déjà existant, skip")
        skipped += 1
    gc.collect()

# COMBINAISONS (10 variantes)
cnt += 1
print(f"[{cnt}] SMOTE + PCA")
if not (CONFIG['skip_existing'] and variant_exists('combined_smote_pca')):
    sm = SMOTE(random_state=42, k_neighbors=3)
    Xs, ys = sm.fit_resample(X_train, y_train)
    if not isinstance(Xs, pd.DataFrame): Xs = pd.DataFrame(Xs, columns=X_train.columns)
    if not isinstance(ys, pd.Series): ys = pd.Series(ys, name='isFraud')
    Xs_sc, _, _ = scale_features(Xs, X_val, X_test)
    p = PCA(8, random_state=42)
    Xp = pd.DataFrame(p.fit_transform(Xs_sc), columns=[f'PC{i+1}' for i in range(8)])
    Vp = pd.DataFrame(p.transform(X_val_scaled), columns=[f'PC{i+1}' for i in range(8)], index=X_val_scaled.index)
    Tp = pd.DataFrame(p.transform(X_test_scaled), columns=[f'PC{i+1}' for i in range(8)], index=X_test_scaled.index)
    save_dataset(Xp, ys, Vp, y_val, Tp, y_test, 'combined_smote_pca',
        {'sampling': 'SMOTE', 'feature_selection': 'none', 'dim_reduction': 'PCA'})
    generated += 1
    del sm, p, Xs, ys, Xs_sc, Xp, Vp, Tp
else:
    print(f"⏭️ combined_smote_pca: Déjà existant, skip")
    skipped += 1
gc.collect()

cnt += 1
print(f"[{cnt}] Undersampling + MI")
if not (CONFIG['skip_existing'] and variant_exists('combined_undersampling_mi')):
    ru = RandomUnderSampler(random_state=42)
    Xr, yr = ru.fit_resample(X_train, y_train)
    if not isinstance(Xr, pd.DataFrame): Xr = pd.DataFrame(Xr, columns=X_train.columns)
    if not isinstance(yr, pd.Series): yr = pd.Series(yr, name='isFraud')
    Xr_sc, _, _ = scale_features(Xr, X_val, X_test)
    sm = SelectKBest(mutual_info_classif, k=10)
    sm.fit(Xr_sc, yr)
    fm = X_train.columns[sm.get_support()].tolist()
    save_dataset(pd.DataFrame(sm.transform(Xr_sc), columns=fm), yr,
        pd.DataFrame(sm.transform(X_val_scaled), columns=fm, index=X_val_scaled.index), y_val,
        pd.DataFrame(sm.transform(X_test_scaled), columns=fm, index=X_test_scaled.index), y_test,
        'combined_undersampling_mi', {'sampling': 'Undersampling', 'feature_selection': 'MI', 'dim_reduction': 'none'})
    generated += 1
    del ru, sm, Xr, yr, Xr_sc
else:
    print(f"⏭️ combined_undersampling_mi: Déjà existant, skip")
    skipped += 1
gc.collect()

cnt += 1
print(f"[{cnt}] SMOTE + LDA")
if not (CONFIG['skip_existing'] and variant_exists('combined_smote_lda')):
    sm = SMOTE(random_state=42, k_neighbors=3)
    Xs, ys = sm.fit_resample(X_train, y_train)
    if not isinstance(Xs, pd.DataFrame): Xs = pd.DataFrame(Xs, columns=X_train.columns)
    if not isinstance(ys, pd.Series): ys = pd.Series(ys, name='isFraud')
    Xs_sc, _, _ = scale_features(Xs, X_val, X_test)
    ld = LDA(n_components=1)
    Xl = pd.DataFrame(ld.fit_transform(Xs_sc, ys), columns=['LD1'])
    Vl = pd.DataFrame(ld.transform(X_val_scaled), columns=['LD1'], index=X_val_scaled.index)
    Tl = pd.DataFrame(ld.transform(X_test_scaled), columns=['LD1'], index=X_test_scaled.index)
    save_dataset(Xl, ys, Vl, y_val, Tl, y_test, 'combined_smote_lda',
        {'sampling': 'SMOTE', 'feature_selection': 'none', 'dim_reduction': 'LDA'})
    generated += 1
    del sm, ld, Xs, ys, Xs_sc, Xl, Vl, Tl
else:
    print(f"⏭️ combined_smote_lda: Déjà existant, skip")
    skipped += 1
gc.collect()

cnt += 1
print(f"[{cnt}] SMOTE + RFE")
if not (CONFIG['skip_existing'] and variant_exists('combined_smote_rfe')):
    sm = SMOTE(random_state=42, k_neighbors=3)
    Xs, ys = sm.fit_resample(X_train, y_train)
    if not isinstance(Xs, pd.DataFrame): Xs = pd.DataFrame(Xs, columns=X_train.columns)
    if not isinstance(ys, pd.Series): ys = pd.Series(ys, name='isFraud')
    Xs_sc, _, _ = scale_features(Xs, X_val, X_test)
    rf = RFE(LogisticRegression(max_iter=500, random_state=42), n_features_to_select = 10)
    rf.fit(Xs_sc, ys)
    fr = X_train.columns[rf.get_support()].tolist()
    save_dataset(pd.DataFrame(rf.transform(Xs_sc), columns=fr), ys,
        pd.DataFrame(rf.transform(X_val_scaled), columns=fr, index=X_val_scaled.index), y_val,
        pd.DataFrame(rf.transform(X_test_scaled), columns=fr, index=X_test_scaled.index), y_test,
        'combined_smote_rfe', {'sampling': 'SMOTE', 'feature_selection': 'RFE', 'dim_reduction': 'none'})
    generated += 1
    del sm, rf, Xs, ys, Xs_sc
else:
    print(f"⏭️ combined_smote_rfe: Déjà existant, skip")
    skipped += 1
gc.collect()

cnt += 1
print(f"[{cnt}] SMOTE + Combined Statistical")
if not (CONFIG['skip_existing'] and variant_exists('combined_smote_statistical')):
    sm = SMOTE(random_state=42, k_neighbors=3)
    Xs, ys = sm.fit_resample(X_train, y_train)
    if not isinstance(Xs, pd.DataFrame): Xs = pd.DataFrame(Xs, columns=X_train.columns)
    if not isinstance(ys, pd.Series): ys = pd.Series(ys, name='isFraud')
    Xs_sc, _, _ = scale_features(Xs, X_val, X_test)
    s1 = SelectKBest(mutual_info_classif, k='all')
    s1.fit(Xs_sc, ys)
    m1 = pd.Series(s1.scores_, index=X_train.columns)
    v1 = Xs_sc.var()
    s2 = SelectKBest(f_classif, k='all')
    s2.fit(Xs_sc, ys)
    a1 = pd.Series(s2.scores_, index=X_train.columns)
    m1n = (m1-m1.min())/(m1.max()-m1.min())
    v1n = (v1-v1.min())/(v1.max()-v1.min())
    a1n = (a1-a1.min())/(a1.max()-a1.min())
    c1 = (m1n+v1n+a1n)/3
    fc = c1.nlargest(10).index.tolist()
    save_dataset(Xs_sc[fc], ys, X_val_scaled[fc], y_val, X_test_scaled[fc], y_test,
        'combined_smote_statistical', {'sampling': 'SMOTE', 'feature_selection': 'Combined', 'dim_reduction': 'none'})
    generated += 1
    del sm, s1, s2, Xs, ys, Xs_sc
else:
    print(f"⏭️ combined_smote_statistical: Déjà existant, skip")
    skipped += 1
gc.collect()

cnt += 1
print(f"[{cnt}] SMOTE + Correlation + PCA")
if not (CONFIG['skip_existing'] and variant_exists('combined_smote_corr_pca')):
    sm = SMOTE(random_state=42, k_neighbors=3)
    Xs, ys = sm.fit_resample(X_train, y_train)
    if not isinstance(Xs, pd.DataFrame): Xs = pd.DataFrame(Xs, columns=X_train.columns)
    if not isinstance(ys, pd.Series): ys = pd.Series(ys, name='isFraud')
    Xs_sc, _, _ = scale_features(Xs, X_val, X_test)
    Xunc, _ = remove_correlated_features(Xs_sc, 0.95)
    nc = min(8, Xunc.shape[1])
    p = PCA(nc, random_state=42)
    Xp = pd.DataFrame(p.fit_transform(Xunc), columns=[f'PC{i+1}' for i in range(nc)])
    Vp = pd.DataFrame(p.transform(X_val_scaled[Xunc.columns]), columns=[f'PC{i+1}' for i in range(nc)], index=X_val_scaled.index)
    Tp = pd.DataFrame(p.transform(X_test_scaled[Xunc.columns]), columns=[f'PC{i+1}' for i in range(nc)], index=X_test_scaled.index)
    save_dataset(Xp, ys, Vp, y_val, Tp, y_test, 'combined_smote_corr_pca',
        {'sampling': 'SMOTE', 'feature_selection': 'Correlation', 'dim_reduction': 'PCA'})
    generated += 1
    del sm, Xs, ys, Xs_sc, Xunc, p, Xp, Vp, Tp
else:
    print(f"⏭️ combined_smote_corr_pca: Déjà existant, skip")
    skipped += 1
gc.collect()

cnt += 1
print(f"[{cnt}] Undersampling + Tree + LDA")
if not (CONFIG['skip_existing'] and variant_exists('combined_under_tree_lda')):
    ru = RandomUnderSampler(random_state=42)
    Xr, yr = ru.fit_resample(X_train, y_train)
    if not isinstance(Xr, pd.DataFrame): Xr = pd.DataFrame(Xr, columns=X_train.columns)
    if not isinstance(yr, pd.Series): yr = pd.Series(yr, name='isFraud')
    Xr_sc, _, _ = scale_features(Xr, X_val, X_test)
    et = ExtraTreesClassifier(n_estimators=30, max_depth=8, random_state=42, n_jobs=-1)
    et.fit(Xr_sc, yr)
    top = pd.DataFrame({'f': X_train.columns, 'i': et.feature_importances_}).nlargest(15, 'i')['f'].tolist()
    ld = LDA(n_components=1)
    Xl = pd.DataFrame(ld.fit_transform(Xr_sc[top], yr), columns=['LD1'])
    Vl = pd.DataFrame(ld.transform(X_val_scaled[top]), columns=['LD1'], index=X_val_scaled.index)
    Tl = pd.DataFrame(ld.transform(X_test_scaled[top]), columns=['LD1'], index=X_test_scaled.index)
    save_dataset(Xl, yr, Vl, y_val, Tl, y_test, 'combined_under_tree_lda',
        {'sampling': 'Undersampling', 'feature_selection': 'ExtraTrees', 'dim_reduction': 'LDA'})
    generated += 1
    del ru, et, Xr, yr, Xr_sc, ld, Xl, Vl, Tl
else:
    print(f"⏭️ combined_under_tree_lda: Déjà existant, skip")
    skipped += 1
gc.collect()

cnt += 1
print(f"[{cnt}] ADASYN + Permutation + PCA")
if not (CONFIG['skip_existing'] and variant_exists('combined_adasyn_perm_pca')):
    ad = ADASYN(random_state=42, n_neighbors=3)
    Xa, ya = ad.fit_resample(X_train, y_train)
    if not isinstance(Xa, pd.DataFrame): Xa = pd.DataFrame(Xa, columns=X_train.columns)
    if not isinstance(ya, pd.Series): ya = pd.Series(ya, name='isFraud')
    Xa_sc, _, _ = scale_features(Xa, X_val, X_test)
    rf = RandomForestClassifier(n_estimators=20, max_depth=6, random_state=42, n_jobs=-1)
    rf.fit(Xa_sc, ya)
    perm = permutation_importance(rf, Xa_sc, ya, n_repeats=3, random_state=42, n_jobs=-1)
    top_p = pd.Series(perm.importances_mean, index=X_train.columns).nlargest(15).index.tolist()
    nc = min(8, len(top_p))
    p = PCA(nc, random_state=42)
    Xp = pd.DataFrame(p.fit_transform(Xa_sc[top_p]), columns=[f'PC{i+1}' for i in range(nc)])
    Vp = pd.DataFrame(p.transform(X_val_scaled[top_p]), columns=[f'PC{i+1}' for i in range(nc)], index=X_val_scaled.index)
    Tp = pd.DataFrame(p.transform(X_test_scaled[top_p]), columns=[f'PC{i+1}' for i in range(nc)], index=X_test_scaled.index)
    save_dataset(Xp, ya, Vp, y_val, Tp, y_test, 'combined_adasyn_perm_pca',
        {'sampling': 'ADASYN', 'feature_selection': 'Permutation', 'dim_reduction': 'PCA'})
    generated += 1
    del ad, Xa, ya, Xa_sc, rf, perm, p, Xp, Vp, Tp
else:
    print(f"⏭️ combined_adasyn_perm_pca: Déjà existant, skip")
    skipped += 1
gc.collect()

cnt += 1
print(f"[{cnt}] SMOTE + L1 + SVD")
if not (CONFIG['skip_existing'] and variant_exists('combined_smote_l1_svd')):
    sm = SMOTE(random_state=42, k_neighbors=3)
    Xs, ys = sm.fit_resample(X_train, y_train)
    if not isinstance(Xs, pd.DataFrame): Xs = pd.DataFrame(Xs, columns=X_train.columns)
    if not isinstance(ys, pd.Series): ys = pd.Series(ys, name='isFraud')
    Xs_sc, _, _ = scale_features(Xs, X_val, X_test)
    las = LassoCV(cv=3, random_state=42, max_iter=500)
    las.fit(Xs_sc, ys)
    top_l1 = pd.Series(np.abs(las.coef_), index=X_train.columns).nlargest(15).index.tolist()
    nc = min(8, len(top_l1))
    sv = TruncatedSVD(nc, random_state=42)
    Xsv = pd.DataFrame(sv.fit_transform(Xs_sc[top_l1]), columns=[f'SVD{i+1}' for i in range(nc)])
    Vsv = pd.DataFrame(sv.transform(X_val_scaled[top_l1]), columns=[f'SVD{i+1}' for i in range(nc)], index=X_val_scaled.index)
    Tsv = pd.DataFrame(sv.transform(X_test_scaled[top_l1]), columns=[f'SVD{i+1}' for i in range(nc)], index=X_test_scaled.index)
    save_dataset(Xsv, ys, Vsv, y_val, Tsv, y_test, 'combined_smote_l1_svd',
        {'sampling': 'SMOTE', 'feature_selection': 'L1_Lasso', 'dim_reduction': 'SVD'})
    generated += 1
    del sm, Xs, ys, Xs_sc, las, sv, Xsv, Vsv, Tsv
else:
    print(f"⏭️ combined_smote_l1_svd: Déjà existant, skip")
    skipped += 1
gc.collect()

cnt += 1
print(f"[{cnt}] SMOTETomek + RFE + PCA")
if not (CONFIG['skip_existing'] and variant_exists('combined_smotetomek_rfe_pca')):
    st = SMOTETomek(random_state=42)
    Xst, yst = st.fit_resample(X_train, y_train)
    if not isinstance(Xst, pd.DataFrame): Xst = pd.DataFrame(Xst, columns=X_train.columns)
    if not isinstance(yst, pd.Series): yst = pd.Series(yst, name='isFraud')
    Xst_sc, _, _ = scale_features(Xst, X_val, X_test)
    rf = RFE(LogisticRegression(max_iter=300, random_state=42), n_features_to_select=12)
    rf.fit(Xst_sc, yst)
    feat_rf = X_train.columns[rf.get_support()].tolist()
    nc = min(8, len(feat_rf))
    p = PCA(nc, random_state=42)
    Xp = pd.DataFrame(p.fit_transform(Xst_sc[feat_rf]), columns=[f'PC{i+1}' for i in range(nc)])
    Vp = pd.DataFrame(p.transform(X_val_scaled[feat_rf]), columns=[f'PC{i+1}' for i in range(nc)], index=X_val_scaled.index)
    Tp = pd.DataFrame(p.transform(X_test_scaled[feat_rf]), columns=[f'PC{i+1}' for i in range(nc)], index=X_test_scaled.index)
    save_dataset(Xp, yst, Vp, y_val, Tp, y_test, 'combined_smotetomek_rfe_pca',
        {'sampling': 'SMOTETomek', 'feature_selection': 'RFE', 'dim_reduction': 'PCA'})
    generated += 1
    del st, Xst, yst, Xst_sc, rf, p, Xp, Vp, Tp
else:
    print(f"⏭️ combined_smotetomek_rfe_pca: Déjà existant, skip")
    skipped += 1
gc.collect()

print(f"\n🎉 Terminé ! Généré: {generated}, Skippé: {skipped}, Total: {cnt} variantes")


🚀 GÉNÉRATION DE TOUTES LES VARIANTES

[1] Baseline
⏭️  baseline: Déjà existant, skip
[2] undersampling_random
⏭️ undersampling_random: Déjà existant, skip
[3] undersampling_tomek
⏭️ undersampling_tomek: Déjà existant, skip
[4] undersampling_nearmiss
⏭️ undersampling_nearmiss: Déjà existant, skip
[5] undersampling_onesided
⏭️ undersampling_onesided: Déjà existant, skip
[6] undersampling_ncr
⏭️ undersampling_ncr: Déjà existant, skip
[7] oversampling_random
⏭️ oversampling_random: Déjà existant, skip
[8] oversampling_smote
⏭️ oversampling_smote: Déjà existant, skip
[9] oversampling_adasyn
⏭️ oversampling_adasyn: Déjà existant, skip
[10] oversampling_borderline
⏭️ oversampling_borderline: Déjà existant, skip
[11] combined_smote_tomek
⏭️ combined_smote_tomek: Déjà existant, skip
[12] combined_smote_enn
⏭️ combined_smote_enn: Déjà existant, skip
[13] ExtraTrees
⏭️ feature_selection_extratrees: Déjà existant, skip
[14] RandomForest
⏭️ feature_selection_randomforest: Déjà existant, skip
[15] 

In [59]:
print("\n📊 RÉSUMÉ FINAL\n")
vlist = sorted([d for d in os.listdir(CONFIG['output_dir']) if os.path.isdir(os.path.join(CONFIG['output_dir'], d))])
summary = []
for v in vlist:
    mpath = os.path.join(CONFIG['output_dir'], v, 'metadata.json')
    if os.path.exists(mpath):
        with open(mpath) as f: m = json.load(f)
        ts = m['shapes']['train']
        fc = m['fraud_distribution']['train'].get('1', 0)
        summary.append({
            'Variant': v,
            'Sampling': m['sampling'],
            'FeatureSel': m['feature_selection'],
            'DimRed': m['dim_reduction'],
            'Shape': f"{ts[0]}x{ts[1]}",
            'Fraud%': f"{fc/ts[0]*100:.1f}%"
        })
df_sum = pd.DataFrame(summary)
print(df_sum.to_string(index=False))
df_sum.to_csv(os.path.join(CONFIG['output_dir'], 'summary.csv'), index=False)
print(f"\n✅ {len(vlist)} variantes dans {CONFIG['output_dir']}")
print(f"📄 summary.csv: {CONFIG['output_dir']}/summary.csv")
print("\n💡 Pour activer Kernel PCA: CONFIG['enable_kernel_pca'] = True")
print("💡 Pour regénérer tout: CONFIG['skip_existing'] = False")


📊 RÉSUMÉ FINAL

                        Variant                Sampling           FeatureSel       DimRed     Shape Fraud%
                       baseline                    none                  all         none 369229x25   5.2%
       combined_adasyn_perm_pca                  ADASYN          Permutation          PCA  703859x9  50.3%
        combined_smote_corr_pca                   SMOTE          Correlation          PCA  700000x9  50.0%
             combined_smote_enn      combined_smote_enn                  all         none 552714x25  57.0%
          combined_smote_l1_svd                   SMOTE             L1_Lasso          SVD  700000x9  50.0%
             combined_smote_lda                   SMOTE                 none          LDA  700000x2  50.0%
             combined_smote_pca                   SMOTE                 none          PCA  700000x9  50.0%
             combined_smote_rfe                   SMOTE                  RFE         none 700000x11  50.0%
     combined_smote_